In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models
import tensorflow as tf

# Step 1: Load the ETTh1 Dataset
url = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"
df = pd.read_csv(url)

# Display dataset information
print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (17420, 8)
                  date   HUFL   HULL   MUFL   MULL   LUFL   LULL         OT
0  2016-07-01 00:00:00  5.827  2.009  1.599  0.462  4.203  1.340  30.531000
1  2016-07-01 01:00:00  5.693  2.076  1.492  0.426  4.142  1.371  27.787001
2  2016-07-01 02:00:00  5.157  1.741  1.279  0.355  3.777  1.218  27.787001
3  2016-07-01 03:00:00  5.090  1.942  1.279  0.391  3.807  1.279  25.044001
4  2016-07-01 04:00:00  5.358  1.942  1.492  0.462  3.868  1.279  21.948000


In [2]:
# Step 2: Normalize the Features
features = ['HUFL', 'HULL', 'MUFL', 'MULL', 'OT']
scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])

# Convert to NumPy array
data = df[features].values

In [3]:
# Step 3: Create Sequences
num_timestamps = 50  # Number of past timestamps to use as input
X, y = [], []

for i in range(len(data) - num_timestamps):
    X.append(data[i:i+num_timestamps, :-1])  # Input: All features except 'OT'
    y.append(data[i+num_timestamps, -1])    # Target: 'OT'

X = np.array(X)
y = np.array(y)

print("Input Shape:", X.shape)  # [Batch size, Number of timestamps, Features]
print("Target Shape:", y.shape)  # [Batch size]

Input Shape: (17370, 50, 4)
Target Shape: (17370,)


In [4]:
# Step 4: Positional Encoding Function
def positional_encoding(seq_len, feature_dim):
    """Generates positional encodings."""
    position = np.arange(seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, feature_dim, 2) * -(np.log(10000.0) / feature_dim))
    pe = np.zeros((seq_len, feature_dim))
    pe[:, 0::2] = np.sin(position * div_term)  # Apply sine to even indices
    pe[:, 1::2] = np.cos(position * div_term)  # Apply cosine to odd indices
    return tf.convert_to_tensor(pe, dtype=tf.float32)

# Add positional encoding
feature_dim = 512
positional_encoding_tensor = positional_encoding(num_timestamps, feature_dim)
print("Positional Encoding Shape:", positional_encoding_tensor.shape)


Positional Encoding Shape: (50, 512)


In [5]:
# Step 5: Define the Transformer Model with Positional Encoding
def build_transformer_model(input_shape, positional_encoding):
    inputs = layers.Input(shape=input_shape)
    x = layers.Dense(feature_dim, activation='relu')(inputs)
    x += positional_encoding  # Add positional encoding
    # Multi-head attention
    x = layers.MultiHeadAttention(num_heads=8, key_dim=64)(x, x)
    x = layers.LayerNormalization()(x)
    # Feedforward network
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(1, activation='linear')(x)  # Predict target variable (OT)
    return models.Model(inputs, x)

# Build and compile the model
transformer_model = build_transformer_model((num_timestamps, X.shape[2]), positional_encoding_tensor)
transformer_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
transformer_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 50, 4)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 50, 512)        │          2,560 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add (Add)                 │ (None, 50, 512)        │              0 │ dense[0][0]            │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ multi_head_attention      │ (None, 50, 512)        │      1,050,624 │ add[0][0], add[0][0]   │
│ (MultiHeadAttention)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ layer_normalization       │ (None, 50, 512)        │          1,024 │ multi_head_attention[… │
│ (LayerNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ global_average_pooling1d  │ (None, 512)            │              0 │ layer_normalization[0… │
│ (GlobalAveragePooling1D)  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 1)              │            513 │ global_average_poolin… │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 1,054,721 (4.02 MB)

 Trainable params: 1,054,721 (4.02 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# Step 6: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 7: Train the Model
history = transformer_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
435/435 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - loss: 8.5145 - mae: 0.9395 - val_loss: 0.0284 - val_mae: 0.1338
Epoch 2/10
435/435 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0307 - mae: 0.1372 - val_loss: 0.0278 - val_mae: 0.1286
Epoch 3/10
435/435 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0293 - mae: 0.1339 - val_loss: 0.0265 - val_mae: 0.1239
Epoch 4/10
435/435 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.0257 - mae: 0.1257 - val_loss: 0.0326 - val_mae: 0.1406
Epoch 5/10
435/435 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.0251 - mae: 0.1215 - val_loss: 0.0451 - val_mae: 0.1732
Epoch 6/10
435/435 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0284 - mae: 0.1300 - val_loss: 0.0400 - val_mae: 0.1675
Epoch 7/10
435/435 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.0275 - mae: 0.1270 - val_loss: 0.0254 - val_mae: 0.1268
Epoch 8/10
435/435 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0251 - mae: 0.1211 - val_loss: 0.0186 - val_mae: 0.1012
Epoch 9/10
435/435 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - 

In [7]:
# Step 8: Evaluate the Model
loss, mae = transformer_model.evaluate(X_test, y_test)
print(f"Test Loss: {loss:.4f}, Test MAE: {mae:.4f}")

# Step 9: Make Predictions
predictions = transformer_model.predict(X_test[:10])
print("Sample Predictions:", predictions.flatten())
print("True Values:", y_test[:10])

109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0164 - mae: 0.0958
Test Loss: 0.0172, Test MAE: 0.0972
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 717ms/step
Sample Predictions: [0.25908357 0.3289463  0.4925437  0.2630993  0.53979087 0.28112757
 0.48130155 0.25260225 0.48763585 0.45499492]
True Values: [0.33146326 0.16573163 0.66572164 0.16010142 0.7780861  0.26543814
 0.54071915 0.52386848 0.66292652 0.49999   ]
